In [7]:
import os
from google.cloud import bigquery
from dotenv import load_dotenv
import vertexai
from google.cloud import discoveryengine_v1 as discoveryengine
from vertexai import generative_models as genai  # Añadir esta línea
from vertexai.generative_models import (
    FunctionDeclaration,
    GenerationConfig,
    Tool,
)

In [2]:
load_dotenv()  # Carga las variables desde .env al entorno
client = bigquery.Client(project='dataton-2024-team-01-cofares')
# Ahora puedes acceder a las variables de entorno
project_id = os.getenv("GOOGLE_CLOUD_PROJECT")

In [3]:
# Configuración del cliente de Vertex AI
PROJECT_ID = "dataton-2024-team-01-cofares"
LOCATION = "us-central1"
vertexai.init(project=PROJECT_ID, location=LOCATION)
#multimodal_model = GenerativeModel("gemini-1.5-flash-001")

# Inicializa el cliente de Discovery Engine
discovery_client = discoveryengine.RankServiceClient()  

In [4]:

def get_products(prompt):
    client = bigquery.Client(project=project_id)
    query = """
    WITH QueryEmbedding AS (
      SELECT
        ml_generate_embedding_result AS query_embedding
      FROM
        ML.GENERATE_EMBEDDING(
          MODEL `dataton-2024-team-01-cofares.datos_cofares.text_embedding`,
          (SELECT @prompt AS content),  -- Aquí usamos el parámetro
          STRUCT(TRUE AS flatten_json_output, 'RETRIEVAL_QUERY' AS task_type)
        )
    )
    SELECT
      d.nombre_completo_material AS nombre,
      d.txt_mas_informacion_del_producto AS descripcion,
      d.txt_instrucciones_de_uso AS modo_implementacion,
      d.codigo_web,
      d.URI_primera_imagen,
      d.codigo_nacional,
      ML.DISTANCE(
        qe.query_embedding,
        e.ml_generate_embedding_result,
        'COSINE'
      ) AS distance_to_query
    FROM
      `dataton-2024-team-01-cofares.datos_cofares.data_final_temp` AS d
    INNER JOIN
      `dataton-2024-team-01-cofares.datos_cofares.SalidaEmbeddings_temp` AS e
      ON d.codigo_web = e.title
    INNER JOIN QueryEmbedding AS qe
      ON TRUE
    ORDER BY
      distance_to_query
    LIMIT 10;
    """.format(prompt)
    # Configura el parámetro para el prompt
    job_config = bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ScalarQueryParameter("prompt", "STRING", prompt)
        ]
    )

    query_job = client.query(query, job_config=job_config)
    results = query_job.result()
    
    products = []
    for row in results:

        descripcion = row.descripcion
        if not row.descripcion:
            descripcion = '-'
        
        modo_implementacion = row.modo_implementacion
        if not row.modo_implementacion:
            modo_implementacion = '-'


        # Cambia la URL si es necesario
        imagen_url = row.URI_primera_imagen 
        if imagen_url and imagen_url.startswith('gs:/'):
            imagen_url = imagen_url.replace('gs://dataton-2024-team-01-cofares-datastore/imagenes/', 'https://storage.googleapis.com/dataton-2024-team-01-cofares-datastore/imagenes/reto_cofares/')
        products.append({
            "codigo_web": row.codigo_web,
            "nombre": row.nombre,
            "codigo_nacional": row.codigo_nacional,
            "descripcion": descripcion,
            "modo_implementacion": modo_implementacion,
            "imagen_url": imagen_url,
            "distance_to_query": row.distance_to_query
        })
    return products

In [5]:
def rerank_products(prompt, products):
    ranking_config = discovery_client.ranking_config_path(
        project=PROJECT_ID,
        location=LOCATION,
        ranking_config="default_ranking_config",
    )
    
    records = [
        discoveryengine.RankingRecord(
            id=str(index),
            title=product["nombre"],
            content=product["descripcion"] + " " + product["modo_implementacion"]
        )
        for index, product in enumerate(products)
    ]
    
    request = discoveryengine.RankRequest(
        ranking_config=ranking_config,
        model="semantic-ranker-512@latest",
        top_n=10, # cantidad de productos a rankear
        query=prompt,
        records=records,
    )
    
    response = discovery_client.rank(request=request)
    
    # Aseguramos que los productos están formateados según el esquema
    ranked_products = [
        {
            "codigo_web": products[int(record.id)]["codigo_web"],
            "nombre": products[int(record.id)]["nombre"],
            "codigo_nacional": products[int(record.id)]["codigo_nacional"],
            "descripcion": products[int(record.id)]["descripcion"],
            "modo_implementacion": products[int(record.id)]["modo_implementacion"],
            "imagen_url": products[int(record.id)]["imagen_url"],
            "distance_to_query": products[int(record.id)]["distance_to_query"]
        }
        for record in response.records[:5] # cantidad de productos a mostrar
    ]
    
    return {"products": ranked_products}

In [9]:
#FUNCTION CALLING

# Define el schema
product_schema = FunctionDeclaration(
    name="product_query",
    description="Fetches relevant product information based on a search prompt.",
    parameters={
        "type": "object",
        "properties": {
            "products": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "codigo_web": {"type": "string", "description": "Product web code"},
                        "nombre": {"type": "string", "description": "Product name"},
                        "codigo_nacional": {"type": "string", "description": "National product code"},
                        "descripcion": {"type": "string", "description": "Product description"},
                        "modo_implementacion": {"type": "string", "description": "Mode of implementation"},
                        "imagen_url": {"type": "string", "description": "Image URL"},
                        "distance_to_query": {"type": "number", "description": "Semantic distance to query"}
                    }
                }
            }
        }
    }
)

# Define tools antes de inicializar el modelo
tools = [Tool(function_declarations=[product_schema])]

In [10]:
PROJECT_ID = "dataton-2024-team-01-cofares"  # @param {type:"string"}
LOCATION = "us-central1"  # @param {type:"string"}

# Importa el modelo de Gemini Flash 1.5
import vertexai
vertexai.init(project=PROJECT_ID, location=LOCATION)


# Model definition
multimodal_model = genai.GenerativeModel(
"gemini-1.5-flash",
generation_config=GenerationConfig(temperature=0),
tools=tools)

chat = multimodal_model.start_chat(response_validation=False)

In [11]:
def generate_response(prompt):  # Eliminamos el parámetro products
    #chat = multimodal_model.start_chat()

    instruction_prompt = f"""
    Eres un asistente farmacéutico experto. 
    
    Consulta recibida de un farmacéutico: "{prompt}"
    
    Por favor, responde de la siguiente manera:
    
    - Saluda a los usuarios y pregúntales en qué puedes ayudarles hoy.
    - Resume la petición del usuario y pídale que confirme que ha entendido correctamente.
    - Si es necesario, pida detalles aclaratorios.
    - Utilice ${tools} para recibir un listado de productos rankeados para ayudar al usuario con su tarea.
    - Agradezca al usuario su colaboración y despídase.
    """

    try:
        response = chat.send_message(instruction_prompt)
        response.candidates[0].content.parts[0]
        
        # Verificar si hay una llamada a función
        for candidate in response.candidates:
            for part in candidate.content.parts:
                if hasattr(part, 'function_call') and part.function_call:
                    # Ejecutar búsqueda de productos
                    products = get_products(prompt)
                    if not products:
                        return "Lo siento, no encontré productos que coincidan con tu búsqueda."
                    
                    ranked_products = rerank_products(prompt, products)
                    
                    # Enviar los resultados al modelo para generar una respuesta contextual
                    results_prompt = f"""
                    Basado en la búsqueda "{prompt}", he encontrado estos productos:
                    {[product['nombre'] for product in ranked_products['products']]}
                    
                    Por favor, genera una respuesta útil que:
                    1. Mencione los productos encontrados
                    2. Explique por qué son relevantes
                    3. Proporcione recomendaciones de uso
                    """
                    
                    final_response = chat.send_message(results_prompt)
                    return {
                        "type": "product_search",
                        "message": final_response.text,
                        "products": ranked_products["products"]
                    }
                
        # Si no hay llamada a función, devolver la respuesta conversacional
        return {
            "type": "conversation",
            "message": response.text
        }
                    
    except Exception as e:
        return f"Lo siento, ocurrió un error: {str(e)}"

In [17]:
# Ejemplo de uso
prompt = "hola"

In [12]:

products = get_products(prompt)  # Llamar a la función para obtener productos
# Imprimir los productos obtenidos
print("Productos obtenidos:")
for product in products:
    print(f"Nombre: {product['nombre']}, Descripción: {product['descripcion']}, Modo de implementación: {product['modo_implementacion']}, Distancia: {product['distance_to_query']}")

Productos obtenidos:
Nombre: Novadiet Colestia Evo, 30 Cápsulas, Descripción: -, Modo de implementación: -, Distancia: 0.31076668748346614
Nombre: Novadiet Colestia Evo, 60 Cápsulas, Descripción: -, Modo de implementación: -, Distancia: 0.3125097921950818
Nombre: INFUSION COL FITOSOL 20FIL, Descripción: INFUSIÓN COL FITOSOL. Contiene 20 bolsitas. Mezcla de plantas para infusión; espino blanco, fumaria y ortosifón., Modo de implementación: -, Distancia: 0.37046435883110995
Nombre: LEVADURA ROJA DE ARROZ 2,9 MILIGRAMOS 30CAPSULAS, Descripción: -, Modo de implementación: -, Distancia: 0.3730689477538758
Nombre: ERGY -3 180 PERLAS, Descripción: -, Modo de implementación: -, Distancia: 0.37333893338898916
Nombre: SLINE CONTROL NOCH BEAUT PLUS+, Descripción: Sline Control Noche Beauty Plus+ es un complemento alimenticio que ayuda a regular la grasa acumulada. y contribuye al control del peso corporal mientras duermes., Modo de implementación: Se recomienda tomar directamente 30ml al día (sal

In [14]:
# Llamar a la función de reranking
ranked_products = rerank_products(prompt, products)["products"]  # Accede a la lista de productos

# Imprimir los productos rankeados
print("Productos rankeados:")
for product in ranked_products:
    print(f"Nombre: {product['nombre']}, Distancia: {product['distance_to_query']}")

Productos rankeados:
Nombre: INFUSION COL FITOSOL 20FIL, Distancia: 0.37046435883110995
Nombre: ESPINO AJO Y OLIVO 60 PERLAS, Distancia: 0.39479307789749085
Nombre: SLINE CONTROL NOCH BEAUT PLUS+, Distancia: 0.39397810963633884
Nombre: OMEGA 3 TG-OMEGA 6 100 PERLAS, Distancia: 0.39486195172190364
Nombre: OMEGA 3-6-9 2960 MG 60 PERLAS, Distancia: 0.40573784475131625


In [18]:
response_text = generate_response(prompt)  # Generar la respuesta
print(response_text)

{'type': 'product_search', 'message': '¡Hola! 👋  Parece que estás buscando información sobre algunos productos. He encontrado estos en mi base de datos:\n\n* **Flor de Bach 15 Holló, 20 ml:** Este producto es un remedio floral que puede ayudar a aliviar el estrés, la ansiedad y la falta de confianza. Se recomienda tomar unas gotas debajo de la lengua varias veces al día, según las necesidades.\n* **HOLOMEGA MELATONINA RETARD 50C:** Este complemento alimenticio contiene melatonina, una hormona que regula el ciclo del sueño. Puede ser útil para combatir el insomnio y mejorar la calidad del sueño. Se recomienda tomar una cápsula al día, antes de acostarse.\n* **Perlapharma Gel Hidroalcohólico Cosmético, 200 ml:** Este gel es un desinfectante de manos que puede ayudar a eliminar bacterias y virus. Se recomienda aplicarlo en las manos y frotar hasta que se seque.\n* **Duplo Aquilea Colágeno + Magnesio, 2 x 375 g:** Este producto es un complemento alimenticio que contiene colágeno y magnesio